``This notebook demonstrates how harmonized datasets are unified.``

``Unification is done across three components:``

``  - CGM data``   
``  - Clinical data (laboratory measurements and clinical visit timepoints)``   
``  - Behavioral data (food and physical activity)``   

In [1]:
import sys
import os

parent_dir = os.path.abspath("../")
sys.path.append(parent_dir)

import polars as pl
from chronoindex.dataset_loader import load_datasets
from chronoindex.dataset_unifier import UnifiedCGMDataset, UnifiedClinicalData, UnifiedFoodData, UnifiedPhysicalActivityData

In [2]:
datasets, errors = load_datasets(
    config_path= "datasets.toml",
    #only=["aleppo","colas","zhao"],
)
if errors:
    raise RuntimeError(errors)

➡️ Loading datasets.aleppo
➡️ Loading datasets.colas
➡️ Loading datasets.zhao
➡️ Loading datasets.praes_unil
➡️ Loading datasets.praes_mug
Error validating item with id s25471121: 2 validation errors for PSubject
visit3Form
  Input should be a valid dictionary or instance of Visit3Form [type=model_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type
visit3AnalysisForm
  Input should be a valid dictionary or instance of VisitAnalysisResultForm [type=model_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type item number 54 in the raw data, data: {'participantId': 's25471121', 'activities': [{'data': {'vo2': {'vo2Max': '45-49'}, 'spo2': None, 'activeZones': None, 'activities': None, 'activitySummary': {'Summary': {'Steps': 251, 'Floors': None, 'Distances': [{'Activity': 'total', 'Distance': 0.1613}, {'Activity': 'tracker', 'Distance': 0.1613}, {'A

In [3]:
unified = UnifiedCGMDataset(
    datasets
    )

df = unified.unified_data   # same as unified.cgm_data
df.head()


👉 Data Harmonization: The Aleppo dataset provides timestamps only as days relative to enrollment. Absolute calendar dates are not available due to de-identification. During harmonization, a placeholder reference date (1990-01-01) is used as the enrollment date to reconstruct pseudo-datetimes.


Id,CGM,CGMTime,dataset
str,list[f64],list[datetime[μs]],str
"""263""","[89.0, 87.0, … 159.0]","[1989-08-16 03:11:00, 1989-08-16 03:16:00, … 1990-09-28 11:02:35]","""aleppo"""
"""101""","[97.0, 79.0, … 87.0]","[1989-10-28 15:23:56, 1989-10-28 17:05:14, … 1990-08-30 09:12:16]","""aleppo"""
"""105""","[125.0, 125.0, … 124.0]","[1990-02-21 18:59:55, 1990-02-21 19:00:18, … 1990-11-19 09:31:31]","""aleppo"""
"""109""","[224.0, 160.0, … 107.0]","[1990-01-01 19:15:48, 1990-01-01 19:18:07, … 1990-09-16 08:48:25]","""aleppo"""
"""240""","[211.0, 55.0, … 175.0]","[1989-10-12 22:08:23, 1989-10-13 18:05:02, … 1990-11-26 14:39:43]","""aleppo"""


In [4]:
# Optional (subset of datasets)
unified_slim = UnifiedCGMDataset(
    datasets,
    dataset_names=["zhao", "colas"],
)

unified_slim.unified_data.head()

Id,CGM,CGMTime,dataset
str,list[f64],list[datetime[μs]],str
"""2000_0_20201230""","[138.6, 120.6, … 88.2]","[2020-12-30 13:54:00, 2020-12-30 14:09:00, … 2021-01-13 12:24:00]","""zhao"""
"""2001_0_20201102""","[102.6, 97.2, … 136.8]","[2020-11-02 09:40:00, 2020-11-02 09:55:00, … 2020-11-14 20:10:00]","""zhao"""
"""2001_1_20201117""","[151.2, 145.8, … 180.0]","[2020-11-17 09:19:00, 2020-11-17 09:34:00, … 2020-12-01 07:49:00]","""zhao"""
"""2002_0_20210513""","[257.4, 246.6, … 217.8]","[2021-05-13 16:22:00, 2021-05-13 16:37:00, … 2021-05-19 14:52:00]","""zhao"""
"""2003_0_20210615""","[264.6, 262.8, … 180.0]","[2021-06-15 17:20:00, 2021-06-15 17:35:00, … 2021-06-21 19:20:00]","""zhao"""


Adding metadata to the CGM.

In [5]:
unified = UnifiedCGMDataset(
    datasets,
    metadata_columns=["Age", "BMI"],   # or metadata_columns="all"
)

df = unified.unified_data   # same as unified.cgm_data
df.head()

👉 Data Harmonization: The Aleppo dataset provides timestamps only as days relative to enrollment. Absolute calendar dates are not available due to de-identification. During harmonization, a placeholder reference date (1990-01-01) is used as the enrollment date to reconstruct pseudo-datetimes.


Id,CGM,CGMTime,Age,BMI,dataset
str,list[f64],list[datetime[μs]],f64,f64,str
"""263""","[89.0, 87.0, … 159.0]","[1989-08-16 03:11:00, 1989-08-16 03:16:00, … 1990-09-28 11:02:35]",44.0,33.455376,"""aleppo"""
"""101""","[97.0, 79.0, … 87.0]","[1989-10-28 15:23:56, 1989-10-28 17:05:14, … 1990-08-30 09:12:16]",27.0,22.395416,"""aleppo"""
"""105""","[125.0, 125.0, … 124.0]","[1990-02-21 18:59:55, 1990-02-21 19:00:18, … 1990-11-19 09:31:31]",42.0,23.47929,"""aleppo"""
"""109""","[224.0, 160.0, … 107.0]","[1990-01-01 19:15:48, 1990-01-01 19:18:07, … 1990-09-16 08:48:25]",78.0,21.453575,"""aleppo"""
"""240""","[211.0, 55.0, … 175.0]","[1989-10-12 22:08:23, 1989-10-13 18:05:02, … 1990-11-26 14:39:43]",68.0,23.140497,"""aleppo"""


Separately Unifying the other data blocks

In [6]:
# 2) Clinical block (default: all lab + visit-timepoint columns when available, unless columns are specified)
uclinical = UnifiedClinicalData(datasets,
    lab_columns=["HbA1c", "CRP", "visit"],
    visit_timepoint_columns=["VisitTimepoint"],
)
clinical_df = uclinical.unified_data
clinical_df.head()


👉 Data Harmonization: The Aleppo dataset provides timestamps only as days relative to enrollment. Absolute calendar dates are not available due to de-identification. During harmonization, a placeholder reference date (1990-01-01) is used as the enrollment date to reconstruct pseudo-datetimes.


Id,HbA1c,CRP,visit,VisitTimepoint,dataset
str,f64,f64,str,date,str
"""263""",7.0,null,"""Screening Visit""",1989-12-29,"""aleppo"""
"""263""",7.0,null,"""Randomization""",1990-03-23,"""aleppo"""
"""263""",6.7,null,"""Week 13 Visit""",1990-06-15,"""aleppo"""
"""263""",7.3,null,"""Week 26 Visit""",1990-09-28,"""aleppo"""
"""101""",5.9,null,"""Screening Visit""",1989-12-13,"""aleppo"""


In [7]:
# Unification of Behavioral data
ubeh = UnifiedFoodData(datasets)
food_df = ubeh.unified_data

👉 Data Harmonization: The Aleppo dataset provides timestamps only as days relative to enrollment. Absolute calendar dates are not available due to de-identification. During harmonization, a placeholder reference date (1990-01-01) is used as the enrollment date to reconstruct pseudo-datetimes.


In [8]:
food_df.filter(pl.col("dataset").is_in(["praes_unil","praes_mug","zhao"]))

Id,food,FoodTimepoint,meal,calories,carbs,proteins,fats,dataset
str,str,datetime[μs],str,f64,f64,f64,f64,str
"""s24101818""","""""",2024-10-08 00:00:00,"""Breakfast""",0.0,0.0,0.0,0.0,"""praes_mug"""
"""s24101818""","""""",2024-10-09 00:00:00,"""Breakfast""",0.0,0.0,0.0,0.0,"""praes_mug"""
"""s24101818""","""""",2024-10-10 00:00:00,"""Breakfast""",0.0,0.0,0.0,0.0,"""praes_mug"""
"""s24101818""","""""",2024-12-05 00:00:00,"""Breakfast""",0.0,0.0,0.0,0.0,"""praes_mug"""
"""s24101818""","""""",2024-12-06 00:00:00,"""Lunch""",0.0,0.0,0.0,0.0,"""praes_mug"""
…,…,…,…,…,…,…,…,…
"""2099_0_20201116""","""data not available""",2020-11-28 18:02:00,null,null,null,null,null,"""zhao"""
"""2099_0_20201116""","""Ginger juice 100 ml Red date s…",2020-11-29 09:02:00,null,null,null,null,null,"""zhao"""
"""2099_0_20201116""","""Hairy crab 315 g Chinese cabba…",2020-11-29 12:02:00,null,null,null,null,null,"""zhao"""


In [9]:
ubeh = UnifiedPhysicalActivityData(datasets)
food_df = ubeh.unified_data

👉 Data Harmonization: The Aleppo dataset provides timestamps only as days relative to enrollment. Absolute calendar dates are not available due to de-identification. During harmonization, a placeholder reference date (1990-01-01) is used as the enrollment date to reconstruct pseudo-datetimes.


In [10]:
food_df.filter(pl.col("dataset").is_in(["praes_unil","praes_mug","zhao"]))

Id,activity_name,startTime,duration,ActivityTimepoint,dataset
str,str,str,i64,datetime[μs],str
"""s24101818""","""Walk""","""09:29""",1074000,2024-10-07 09:29:00,"""praes_mug"""
"""s24101818""","""Walk""","""10:17""",3224000,2024-10-13 10:17:00,"""praes_mug"""
"""s24101818""","""Walk""","""20:58""",1230000,2024-10-14 20:58:00,"""praes_mug"""
"""s24101818""","""Walk""","""05:54""",974000,2024-12-06 05:54:00,"""praes_mug"""
"""s24101818""","""Walk""","""17:39""",1229000,2024-12-20 17:39:00,"""praes_mug"""
…,…,…,…,…,…
"""s2512892""","""Run""","""19:31""",3549000,2025-06-05 19:31:00,"""praes_unil"""
"""s2512892""","""Walk""","""15:48""",6450000,2025-06-07 15:48:00,"""praes_unil"""
"""s2512892""","""Run""","""10:00""",3518000,2025-06-08 10:00:00,"""praes_unil"""
